In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler


In [2]:
pd.read_csv("netflix_customer_churn.csv").head(10)

#monthly_fee
#watch_hours
#avg_watch_time_per_day


,customer_id,age,gender,subscription_type,watch_hours,last_login_days,region,device,monthly_fee,churned,payment_method,number_of_profiles,avg_watch_time_per_day,favorite_genre
0,a9b75100-82a8-427a-a208-72f24052884a,51,Other,Basic,14.73,29,Africa,TV,8.99,1,Gift Card,1,0.49,Action
1,49a5dfd9-7e69-4022-a6ad-0a1b9767fb5b,47,Other,Standard,0.70,19,Europe,Mobile,13.99,1,Gift Card,5,0.03,Sci-Fi
2,4d71f6ce-fca9-4ff7-8afa-197ac24de14b,27,Female,Standard,16.32,10,Asia,TV,13.99,0,Crypto,2,1.48,Drama
3,d3c72c38-631b-4f9e-8a0e-de103cad1a7d,53,Other,Premium,4.51,12,Oceania,TV,17.99,1,Crypto,2,0.35,Horror
4,4e265c34-103a-4dbb-9553-76c9aa47e946,56,Other,Standard,1.89,13,Africa,Mobile,13.99,1,Crypto,2,0.13,Action
5,d8079475-5be7-47e9-8782-ceb7ff61395e,58,Female,Standard,13.80,26,Oceania,Mobile,13.99,0,Debit Card,3,0.51,Action
6,8e63450a-13d6-4e83-bbb5-6aebde9152cb,48,Other,Basic,13.83,20,Asia,TV,8.99,0,Gift Card,5,0.66,Romance
7,02387681-8c42-462a-807a-de0168c73b38,51,Male,Basic,14.30,56,Europe,Mobile,8.99,1,Gift Card,1,0.25,Action
8,0bcaad0c-545c-4ee1-85a6-75e165f39361,45,Other,Basic,9.98,10,Asia,Mobile,8.99,0,PayPal,3,0.91,Romance
9,eae6439e-8cdf-4258-ab49-c493925b927a,32,Other,Premium,2.22,34,Europe,TV,17.99,1,Debit Card,1,0.06,Drama


In [3]:
datos = pd.read_csv("netflix_customer_churn.csv")

In [4]:
datos = datos.drop(["customer_id"], axis=1)

In [5]:
datos = pd.get_dummies(datos, dtype=int)

In [6]:
datos.head(10)

,age,watch_hours,last_login_days,monthly_fee,churned,number_of_profiles,avg_watch_time_per_day,gender_Female,gender_Male,gender_Other,...,payment_method_Debit Card,payment_method_Gift Card,payment_method_PayPal,favorite_genre_Action,favorite_genre_Comedy,favorite_genre_Documentary,favorite_genre_Drama,favorite_genre_Horror,favorite_genre_Romance,favorite_genre_Sci-Fi
0,51,14.73,29,8.99,1,1,0.49,0,0,1,...,0,1,0,1,0,0,0,0,0,0
1,47,0.70,19,13.99,1,5,0.03,0,0,1,...,0,1,0,0,0,0,0,0,0,1
2,27,16.32,10,13.99,0,2,1.48,1,0,0,...,0,0,0,0,0,0,1,0,0,0
3,53,4.51,12,17.99,1,2,0.35,0,0,1,...,0,0,0,0,0,0,0,1,0,0
4,56,1.89,13,13.99,1,2,0.13,0,0,1,...,0,0,0,1,0,0,0,0,0,0
5,58,13.80,26,13.99,0,3,0.51,1,0,0,...,1,0,0,1,0,0,0,0,0,0
6,48,13.83,20,8.99,0,5,0.66,0,0,1,...,0,1,0,0,0,0,0,0,1,0
7,51,14.30,56,8.99,1,1,0.25,0,1,0,...,0,1,0,1,0,0,0,0,0,0
8,45,9.98,10,8.99,0,3,0.91,0,0,1,...,0,0,1,0,0,0,0,0,1,0
9,32,2.22,34,17.99,1,1,0.06,0,0,1,...,1,0,0,0,0,0,1,0,0,0


In [7]:
X = datos.drop(["churned"], axis=1)
Y = datos["churned"]

In [8]:
X_ent,X_test,Y_ent,Y_test = train_test_split(X,Y, test_size=0.2, random_state=42)


In [9]:
def init_params(n_inputs):
    w1 = np.random.rand(10,n_inputs) - 0.5
    b1 = np.random.rand(10,1) - 0.5
    w2 = np.random.rand(1,10) -0.5
    b2 = np.random.rand(1,1) - 0.5
    return w1,b1,w2,b2

def ReLU(Z):
    return np.maximum(0, Z)

def sigmoid(Z):
    return 1 / (1+np.exp(-Z))

def g_ReLu(Z):
    return Z > 0

def forward(w1,b1,w2,b2, x):
    z1 = w1.dot(x) + b1
    a1 = ReLU(z1)
    z2 = w2.dot(a1) + b2
    a2 = sigmoid(z2)
    return z1,a1,z2,a2

def backprop(z1,a1,a2,w2,x,y):
    m = y.shape[1]
    dz2 = a2 - y
    dw2 = (1/m) * dz2.dot(a1.T)
    db2 = (1/m) * np.sum(dz2, axis=1, keepdims=True)
    dz1 = w2.T.dot(dz2) * g_ReLu(z1)
    dw1 = (1/m) * dz1.dot(x.T)
    db1 = (1/m) * np.sum(dz1, axis=1, keepdims=True)
    return dw1,db1,dw2,db2

def update(w1,b1,w2,b2,dw1,db1,dw2,db2,alpha):
    w1= w1 -alpha*dw1
    b1= b1-alpha*db1
    w2= w2-alpha*dw2
    b2= b2-alpha*db2
    return w1,b1,w2,b2


In [10]:
#X_ent, X_test, Y_ent, Y_test = train_test_split(X,Y,test_size=0.2, random_state=42)

scaler = StandardScaler()

X_ent = scaler.fit_transform(X_ent)
X_test = scaler.transform(X_test)

In [11]:
X_ent = X_ent.T
X_test = X_test.T
Y_ent = np.array(Y_ent).reshape(1, -1)
Y_test = np.array(Y_test).reshape(1, -1)

In [12]:
def get_predict(a2):
    return (a2 >= 0.5).astype(int)

def accuracy(predictions, y):
    return np.sum(predictions == y) / y.size

def g_d(x,y,it, alpha):
    w1,b1,w2,b2 = init_params(x.shape[0])
    for i in range(it):
        z1,a1,z2,a2 = forward(w1,b1, w2,b2,x)
        dw1,db1,dw2,db2 =backprop(z1,a1,a2,w2,x,y)
        w1,b1,w2,b2 = update(w1,b1,w2,b2,dw1,db1,dw2,db2,alpha)
        if i % 500 == 0:
            predictions = get_predict(a2)
            print(f"it: {i}")
            print(f"Accuracy: {accuracy(predictions, y)}")
    return w1,b1,w2,b2

In [13]:
w1,b1,w2,b2=g_d(X_ent,Y_ent,5000,0.01)

it: 0
Accuracy: 0.5495
it: 500
Accuracy: 0.7155
it: 1000
Accuracy: 0.79025
it: 1500
Accuracy: 0.83425
it: 2000
Accuracy: 0.8555
it: 2500
Accuracy: 0.868
it: 3000
Accuracy: 0.875
it: 3500
Accuracy: 0.87975
it: 4000
Accuracy: 0.88625
it: 4500
Accuracy: 0.89375


In [16]:
pd.read_csv("netflix_customer_churn.csv").head(4)

,customer_id,age,gender,subscription_type,watch_hours,last_login_days,region,device,monthly_fee,churned,payment_method,number_of_profiles,avg_watch_time_per_day,favorite_genre
0,a9b75100-82a8-427a-a208-72f24052884a,51,Other,Basic,14.73,29,Africa,TV,8.99,1,Gift Card,1,0.49,Action
1,49a5dfd9-7e69-4022-a6ad-0a1b9767fb5b,47,Other,Standard,0.70,19,Europe,Mobile,13.99,1,Gift Card,5,0.03,Sci-Fi
2,4d71f6ce-fca9-4ff7-8afa-197ac24de14b,27,Female,Standard,16.32,10,Asia,TV,13.99,0,Crypto,2,1.48,Drama
3,d3c72c38-631b-4f9e-8a0e-de103cad1a7d,53,Other,Premium,4.51,12,Oceania,TV,17.99,1,Crypto,2,0.35,Horror


In [19]:
#\-- Predicciones --\

index = 39
X_ejemplo = X_test[:, index].reshape(-1,1)
_,_,_,a2 = forward(w1,b1,w2,b2,X_ejemplo)

prob = a2[0,0]
prediccion = "Si" if prob >= 0.5 else "No"
real = "Yes" if Y_test[0, index] == 1 else "No"


print(f"Prob de irse: {prob*100:.2f} %")
print(f"Pred: {prediccion}")
print(f"Valor Esperado: {real}")

Prob de irse: 77.78 %
Pred: Si
Valor Esperado: Yes
